# ATAT Model Attention Visualization

This notebook demonstrates how to use the new attention visualization functionality added to the ATAT (Astronomical Time-series Attention Transformer) model.

The visualization helps in model interpretability by showing which parts of the light curve the model is focusing on during prediction.

In [3]:
# Import necessary libraries
import sys
sys.path.append('../')

import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ATAT model and attention visualization components
from src.models.components.atat_net import ATAT
from src.models.components.transformer.AttentionVisualization import (
    AttentionExtractor, 
    AttentionVisualizer,
    visualize_model_attention
)

## Load a Pre-trained Model

First, let's load a pre-trained ATAT model from a checkpoint.

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Path to model checkpoint
checkpoint_path = "../logs/train/multiruns/atat_baseline_mm/1/checkpoints/epoch_0093.ckpt"

# Check if checkpoint exists
if not os.path.exists(checkpoint_path):
    print(f"Checkpoint not found at {checkpoint_path}")
    print("Using a freshly initialized model instead. Results will not be meaningful.")
    checkpoint = None
else:
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

# Initialize the ATAT model
# Note: Parameters should match the ones used during training
model_args = {
    "use_metadata": True,
    "use_features": True,
    "use_lightcurve": True,
    "embedding_size_lc": 128,
    "embedding_size_feat": 128,
    "num_metadata": 10,  # Adjust based on your dataset
    "num_features": 10,  # Adjust based on your dataset
    "num_encoders": 3,
    "num_heads": 8,
    "attn_type": "mha",  # Regular multihead attention
    "dropout": 0.1,
    "prenorm": True
}

model = ATAT(**model_args)

# Load state dict if checkpoint is available
if checkpoint is not None:
    if 'state_dict' in checkpoint:
        # Handle Lightning module state_dict format
        state_dict = {}
        for key, value in checkpoint['state_dict'].items():
            if key.startswith('model.'):
                # Remove 'model.' prefix
                state_dict[key[6:]] = value
        model.load_state_dict(state_dict)
    else:
        # Direct state dict
        model.load_state_dict(checkpoint)

# Move model to device and set to evaluation mode
model.to(device)
model.eval()

Using device: cuda
Loading checkpoint from ../logs/train/multiruns/atat_baseline_mm/1/checkpoints/epoch_0093.ckpt


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL omegaconf.listconfig.ListConfig was not an allowed global by default. Please use `torch.serialization.add_safe_globals([ListConfig])` or the `torch.serialization.safe_globals([ListConfig])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default

: 

## Load Test Data

Now we'll load some test data to visualize attention on.

In [ ]:
# Load test data
dataset_path = "/home/fsoto/Documents/LCsSSL/data/alerce_data/final/dataset_with_periods.h5"

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"Dataset not found at {dataset_path}")
    print("Creating dummy data for demonstration...")
    
    # Create dummy data
    batch_size = az1
    seq_len = 100
    data = torch.randn(batch_size, seq_len, 2)
    time = torch.sort(torch.rand(batch_size, seq_len, 1) * 100, dim=1)[0]
    mask = torch.ones(batch_size, seq_len, 1)
    metadata = torch.randn(batch_size, model_args["num_metadata"])
    features = torch.randn(batch_size, model_args["num_features"])
    bands = torch.zeros(batch_size, seq_len, 1)  # All same band for simplicity
else:
    # Load real data from the dataset
    print(f"Loading dataset from {dataset_path}")
    dataset = h5py.File(dataset_path, 'r')
    
    # Select a random sample from the test set
    test_indices = np.where(dataset['partition'][()] == 2)[0]
    sample_idx = np.random.choice(test_indices)
    
    # Extract data for the sample
    lc_data = dataset['data'][sample_idx]
    lc_time = dataset['time'][sample_idx]
    lc_mask = dataset['mask'][sample_idx]
    bands_data = dataset['fid'][sample_idx]
    
    # Normalize and prepare data
    lc_data = (lc_data - np.mean(lc_data[lc_mask[:, 0] > 0])) / np.std(lc_data[lc_mask[:, 0] > 0])
    lc_time = lc_time - np.min(lc_time[lc_mask[:, 0] > 0])
    
    # Convert to batch format (add batch dimension)
    data = torch.tensor(lc_data).float().unsqueeze(0)
    time = torch.tensor(lc_time).float().unsqueeze(0)
    mask = torch.tensor(lc_mask).float().unsqueeze(0)
    bands = torch.tensor(bands_data).float().unsqueeze(0)
    
    # For metadata and features, check if they exist in the dataset
    if 'metadata' in dataset and 'features' in dataset:
        metadata = torch.tensor(dataset['metadata'][sample_idx]).float().unsqueeze(0)
        features = torch.tensor(dataset['features'][sample_idx]).float().unsqueeze(0)
    else:
        # Create dummy metadata and features if not available
        metadata = torch.randn(1, model_args["num_metadata"])
        features = torch.randn(1, model_args["num_features"])

    # Get class label for this sample
    label = dataset['target'][sample_idx]
    label_name = dataset['classes'][label]
    
    print(f"Selected sample: {sample_idx}, Class: {label_name}")

# Move data to the device
data = data.to(device)
time = time.to(device)
mask = mask.to(device)
metadata = metadata.to(device)
features = features.to(device)
bands = bands.to(device)

## Visualize the Light Curve

Let's first visualize the raw light curve data.

In [ ]:
def plot_light_curve(time, data, mask, bands=None, title=None):
    """Plot a light curve with optional band information."""
    # Get data from first batch
    time_data = time[0].cpu().numpy()
    flux_data = data[0].cpu().numpy()
    mask_data = mask[0].cpu().numpy()
    
    # Apply mask
    valid_indices = mask_data[:, 0] > 0
    time_valid = time_data[valid_indices]
    flux_valid = flux_data[valid_indices]
    
    # Plot the light curve
    plt.figure(figsize=(12, 6))
    
    if bands is not None:
        # Plot separate bands if available
        bands_data = bands[0].cpu().numpy()
        bands_valid = bands_data[valid_indices]
        
        # Identify unique bands
        unique_bands = np.unique(bands_valid)
        colors = ['blue', 'red', 'green', 'purple', 'orange']
        
        for i, band in enumerate(unique_bands):
            band_mask = bands_valid == band
            plt.scatter(time_valid[band_mask], flux_valid[band_mask], 
                       label=f'Band {int(band)}', color=colors[i % len(colors)])
        plt.legend()
    else:
        # Simple plot without band information
        plt.scatter(time_valid, flux_valid, color='blue')
    
    plt.xlabel('Time')
    plt.ylabel('Flux')
    if title:
        plt.title(title)
    else:
        plt.title('Light Curve Data')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

# Plot the sample light curve
plot_light_curve(time, data, mask, bands, title=f"Light Curve (Class: {label_name if 'label_name' in locals() else 'Unknown'})")

## Extract and Visualize Attention

Now, let's extract and visualize the attention weights from the model.

In [ ]:
# Enable attention saving in all attention modules
def enable_attention_saving(model, enable=True):
    """Recursively enable attention weight saving in all attention modules."""
    for module in model.modules():
        if hasattr(module, 'save_attention'):
            module.save_attention = enable
            
# Enable attention saving
enable_attention_saving(model)

In [ ]:
# Initialize the attention extractor
extractor = AttentionExtractor()
extractor.register_hooks(model)

# Forward pass through the model
with torch.no_grad():
    output = model(data=data, time=time, mask=mask, metadata=metadata, features=features, bands=bands)

# Get attention weights
attention_weights = extractor.get_attention_weights()

# Display summary of extracted attention weights
print(f"Extracted attention weights from {len(attention_weights)} layers:")
for layer_name, weights in attention_weights.items():
    print(f"  {layer_name}: {tuple(weights.shape)}")

## Visualize Attention as Heatmaps

Let's visualize the attention weights from different layers as heatmaps.

In [ ]:
# Create visualizer
visualizer = AttentionVisualizer()

# Get a list of layer names we want to visualize
# For simplicity, select a subset of the attention layers if there are many
num_to_show = min(4, len(attention_weights))
layer_names = list(attention_weights.keys())[:num_to_show]

# Plot heatmaps for each selected layer
for layer_name in layer_names:
    weights = attention_weights[layer_name]
    
    # Plot for first head
    head_idx = 0
    fig = visualizer.plot_attention_heatmap(
        attention_weights=weights,
        time_values=time,
        layer_name=layer_name,
        head_idx=head_idx,
        batch_idx=0,
        figsize=(10, 8),
        title=f"{layer_name} - Head {head_idx}"
    )
    plt.show()
    
    # For the first layer, also show multiple heads
    if layer_name == layer_names[0] and weights.shape[1] > 1:
        # Create a grid of multiple attention heads
        num_heads_to_show = min(4, weights.shape[1])
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.flatten()
        
        for i in range(num_heads_to_show):
            ax = axes[i]
            attn = weights[0, i].cpu().numpy()
            im = ax.imshow(attn, cmap='viridis')
            ax.set_title(f"Head {i}")
            fig.colorbar(im, ax=ax)
        
        plt.suptitle(f"Multiple Attention Heads: {layer_name}", fontsize=16)
        plt.tight_layout()
        plt.show()

## Visualize Attention Over Time

Now let's visualize how the attention distributes over the time series data.

In [ ]:
# Plot attention over time for selected layers
for layer_name in layer_names:
    weights = attention_weights[layer_name]
    
    # Plot for the first head, attention from CLS token (token_idx=0)
    fig = visualizer.plot_attention_over_time(
        attention_weights=weights,
        time_values=time,
        flux_values=data,
        layer_name=layer_name,
        head_idx=0,
        token_idx=0,  # CLS token
        batch_idx=0,
        figsize=(12, 6)
    )
    plt.show()

## Using the Integrated Visualization Tool

The module also provides an integrated function `visualize_model_attention` that handles the extraction and visualization in one step.

In [ ]:
# Clear previous hooks
extractor.clear()

# Prepare input data dictionary
input_data = {
    'data': data,
    'time': time,
    'mask': mask,
    'metadata': metadata,
    'features': features,
    'bands': bands
}

# Create output directory for saved figures if needed
output_dir = "../logs/attention_viz"
os.makedirs(output_dir, exist_ok=True)

# Use the integrated visualization function
figures = visualize_model_attention(
    model=model,
    input_data=input_data,
    output_path=output_dir,
    time_key='time',
    data_key='data',
    head_idx=0,
    batch_idx=0,
    token_idx=0 # CLS token
)

print(f"Generated {len(figures)} visualization figures.")
print(f"Figures saved to {output_dir}")

## Analyzing Results

Let's analyze what the attention patterns tell us about the model's focus on different parts of the light curve.

In [ ]:
# Function to find peaks in attention weights
def find_attention_peaks(attention_weights, layer_name, head_idx=0, token_idx=0, batch_idx=0, threshold=0.8):
    """Find time points with high attention focus."""
    # Get weights for the specified layer, head, token
    weights = attention_weights[layer_name][batch_idx, head_idx, token_idx, 1:].cpu().numpy()  # Skip CLS token
    
    # Find points above threshold (relative to max)
    relative_weights = weights / weights.max()
    peak_indices = np.where(relative_weights > threshold)[0]
    
    return peak_indices

# Show most attended time points for each layer
for layer_name in layer_names:
    if layer_name in attention_weights:
        # Find peaks for this layer
        peak_indices = find_attention_peaks(attention_weights, layer_name)
        
        # Get the corresponding time points
        time_data = time[0, 1:].cpu().numpy()  # Skip CLS token
        peak_times = time_data[peak_indices]
        
        # Print results
        print(f"\n{layer_name} - High attention time points:")
        if len(peak_times) > 0:
            for i, t in enumerate(peak_times[:5]):  # Show at most 5 peaks
                print(f"  Peak {i+1}: Time = {t}")
        else:
            print("  No strong attention peaks found")
            
        # Plot the light curve with highlighted attention points
        time_np = time[0].cpu().numpy()
        data_np = data[0].cpu().numpy()
        mask_np = mask[0].cpu().numpy()
        
        # Apply mask
        valid_indices = mask_np[:, 0] > 0
        time_valid = time_np[valid_indices]
        data_valid = data_np[valid_indices]
        
        plt.figure(figsize=(12, 6))
        plt.scatter(time_valid, data_valid, alpha=0.5, label='Light curve')
        
        # Highlight the high attention points
        if len(peak_indices) > 0:
            weight_values = attention_weights[layer_name][0, 0, 0, 1:].cpu().numpy()
            peak_weights = weight_values[peak_indices]
            peak_data = data_np[peak_indices+1]  # +1 to account for CLS token offset
            plt.scatter(peak_times, peak_data, color='red', s=100*peak_weights/peak_weights.max(), 
                       alpha=0.7, label='High attention')
        
        plt.title(f"{layer_name} - Attention Focus")
        plt.xlabel('Time')
        plt.ylabel('Flux')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.show()

## Periodic Attention Analysis

For astronomical data, periodic patterns are often important. Let's analyze the periodic bias in the attention mechanism if using periodic attention.

In [ ]:
# Check if any periodic attention modules are present
periodic_attn_modules = [m for m in model.modules() if hasattr(m, 'get_periodic_bias') and callable(getattr(m, 'get_periodic_bias'))]

if periodic_attn_modules:
    print(f"Found {len(periodic_attn_modules)} periodic attention modules")
    
    # Analyze periodic bias
    for i, module in enumerate(periodic_attn_modules):
        # Get periodic bias
        periodic_bias = module.get_periodic_bias()
        
        if periodic_bias is not None:
            # Visualize the periodic bias
            plt.figure(figsize=(10, 8))
            plt.imshow(periodic_bias[0, 0].cpu().numpy(), cmap='viridis')
            plt.colorbar(label='Periodic Bias')
            plt.title(f"Periodic Attention Bias - Module {i+1}")
            plt.xlabel('Sequence Position')
            plt.ylabel('Sequence Position')
            plt.show()
else:
    print("No periodic attention modules found in the model")

## Conclusion

In this notebook, we've demonstrated how to visualize and analyze attention weights from the ATAT transformer model. These visualizations help us understand which parts of the astronomical time series data the model is focusing on when making predictions.

Key takeaways:

1. Attention heatmaps show the interaction between different positions in the sequence
2. Time-series attention plots reveal which time points are most important for the model
3. Periodic attention bias can indicate how the model handles cyclical patterns in the data

These insights can be valuable for interpreting model predictions and potentially improving the model architecture.